# Narrativa de la comparación de jugadores con LLM

`app/src/app/narrativa/generador.py` (backend de ScoutIA) es el único componente de LLM que
llegó a producción: `NarradorComparacion.redactar()`, servido por
`GET /jugadores/comparar/narrativa`. Reutiliza el patrón `response_format: json_schema` que
se probó en el prototipo original de este notebook (ESPN + OpenAI para clasificar noticias,
descartado el 2026-09-20 -- ver AGENTS.md, sección "Notebook `05-llm.ipynb` eliminado") pero
para un problema distinto: no clasifica texto libre, **redacta en prosa un JSON que la app ya
calculó**.

Regla no negociable del spec de ScoutIA (repetida en AGENTS.md): el LLM nunca calcula un
número, solo verbaliza lo que ya resolvieron los percentiles precalculados, el modelo de
valor (Lasso) y el clustering de arquetipos (KMeans). Este notebook no genera ningún artefacto
nuevo (a diferencia de `03-ingenieria_variables.ipynb`, `04-modelado.ipynb` o
`06-perfil.ipynb`, cuyo trabajo terminó congelado en `scripts/build_model_artifacts.py`) --
el código de `NarradorComparacion` ya está en `app/`, esto documenta y reproduce su diseño
para poder seguir iterando el prompt/schema desde un solo lugar, como el resto de las partes
del proyecto.

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# app/ es un proyecto uv independiente (su propio .venv, ver AGENTS.md "Dos mundos de
# Python") -- no se instala en .venv312. Pero narrativa/, valuation/ y schemas/ no dependen
# de FastAPI (solo de pydantic, joblib, pandas, openai, todos ya en .venv312), así que se
# pueden importar directo agregando app/src al sys.path, sin arrastrar el servidor.
sys.path.insert(0, str(Path("..") / "app" / "src"))

load_dotenv("../.env")  # mismo .env de la raíz que usa Settings en app/src/app/core/config.py
client_api_key = os.getenv("OPENAI_API_KEY")
assert client_api_key, "Falta OPENAI_API_KEY en el .env de la raíz"


## 1. Dos jugadores reales del dataset servido

`data/processed/dataset.parquet` es el mismo panel jugador-temporada que sirve el backend
(`Settings.dataset_path`) -- ya trae los `pctl_*` y `cluster_id`/`cluster_label`
precalculados por `scripts/build_model_artifacts.py`, así que no hay que recalcular
percentiles ni volver a correr el clustering acá.

Se eligen dos delanteros centro de la temporada más reciente con valores de mercado (TM) muy
distintos -- un caso donde vale la pena ver si el modelo + el LLM explican bien la brecha.

In [2]:
df = pd.read_parquet("../data/processed/dataset.parquet")

recientes = df[df["Season"].astype(str) == "2025-2026"]
fila_a = recientes[recientes["Player"] == "Erling Haaland"].iloc[0]
fila_b = recientes[recientes["Player"] == "Kylian Mbappé"].iloc[0]

fila_a[["Player", "Squad", "posicion_tm", "Age", "valor_eur", "cluster_label"]]


Player                                Erling Haaland
Squad                                Manchester City
posicion_tm                           Centre-Forward
Age                                             25.0
valor_eur                                200000000.0
cluster_label    Estrella ofensiva de máximo volumen
Name: 11479, dtype: object

## 2. Reconstruir la comparación con las clases reales de la app

Nada de esto se reimplementa "a mano" con lógica nueva: son las mismas funciones privadas de
`app/src/app/api/v1/jugadores.py` (`_estadisticas_de`, `_grupo_comparacion_de`,
`_percentiles_comparados_de`, `_valor_comparado_de`, `_arquetipo_comparado_de`) que arma
`GET /jugadores/comparar`, copiadas acá sin la capa de `Request`/FastAPI -- todo lo que
calculan (`Predictor.explicar_diferencia`, los `pctl_*`, `cluster_id`) sale de las clases y
columnas reales, no de datos inventados para el demo.

In [3]:
from app.domain.feature_labels import etiqueta_legible
from app.narrativa.generador import RESPONSE_SCHEMA, SYSTEM_PROMPT, NarradorComparacion, _datos_para_prompt
from app.schemas.comparacion import (
    ArquetipoComparado,
    ComparacionJugadores,
    ContribucionComparada,
    NarrativaComparacion,
    PercentilComparado,
    ValorComparado,
)
from app.schemas.jugador import EstadisticasJugador
from app.schemas.percentiles import GrupoComparacion
from app.valuation.predictor import Predictor

PERCENTILE_STATS = [  # mismo subconjunto que api/v1/jugadores.py -- ver AGENTS.md sobre
    "Per 90 Minutes_Gls", "Per 90 Minutes_Ast", "Standard_Sh/90", "Standard_SoT%",
    "Performance_TklW", "Performance_Int", "Performance_Crs", "Playing Time_Min%",
]

predictor = Predictor(Path("../models"))


def _url_de(fila, columna):
    valor = fila.get(columna)
    return str(valor) if pd.notna(valor) else None


def _estadisticas_de(player_id, fila):
    return EstadisticasJugador(
        player_id=player_id, nombre=str(fila["Player"]), club=str(fila["Squad"]),
        nacionalidad=str(fila["Nation"]), posicion=str(fila["Pos"]),
        posicion_detallada=str(fila["posicion_tm"]), liga=str(fila["Comp"]),
        temporada=str(fila["Season"]), foto_url=_url_de(fila, "foto_url"),
        bandera_url=_url_de(fila, "bandera_url"), escudo_url=_url_de(fila, "escudo_url"),
        edad=float(fila["Age"]), minutos_jugados=int(fila["Playing Time_Min"]),
        goles=int(fila["Performance_Gls"]), asistencias=int(fila["Performance_Ast"]),
        tiros=int(fila["Standard_Sh"]), tarjetas_amarillas=int(fila["Performance_CrdY"]),
        valor_mercado_eur=float(fila["valor_eur"]),
        contrato_conocido=bool(fila["flag_contrato_conocido"]),
        dias_contrato_restante=(
            float(fila["dias_contrato_restante"]) if fila["flag_contrato_conocido"] else None
        ),
        gano_liga=bool(fila["Ganador_Liga"]), gano_champions=bool(fila["Ganador_Champions"]),
    )


def _grupo_comparacion_de(fila):
    return GrupoComparacion(
        posicion=str(fila["posicion_tm"]), temporada=str(fila["Season"]),
        n=int(fila["n_comparacion"]),
    )


def _percentiles_comparados_de(fila_a, fila_b):
    return [
        PercentilComparado(
            estadistica=stat, etiqueta=etiqueta_legible(stat),
            percentil_a=float(fila_a[f"pctl_{stat}"]), percentil_b=float(fila_b[f"pctl_{stat}"]),
            valor_bruto_a=float(fila_a[stat]), valor_bruto_b=float(fila_b[stat]),
            diferencia=float(fila_a[f"pctl_{stat}"]) - float(fila_b[f"pctl_{stat}"]),
        )
        for stat in PERCENTILE_STATS
    ]


def _valor_comparado_de(fila_a, fila_b, top_n=8):
    banda_a = predictor.predecir_banda(fila_a)
    banda_b = predictor.predecir_banda(fila_b)
    diferencias = predictor.explicar_diferencia(fila_a, fila_b, top_n=top_n)
    return ValorComparado(
        banda_a=banda_a, banda_b=banda_b,
        diferencia_valor_medio_eur=banda_a["valor_medio"] - banda_b["valor_medio"],
        contribuciones_diferencia=[ContribucionComparada(**c) for c in diferencias],
    )


def _arquetipo_comparado_de(fila_a, fila_b):
    cluster_a, cluster_b = int(fila_a["cluster_id"]), int(fila_b["cluster_id"])
    return ArquetipoComparado(
        cluster_id_a=cluster_a, etiqueta_a=str(fila_a["cluster_label"]),
        cluster_id_b=cluster_b, etiqueta_b=str(fila_b["cluster_label"]),
        mismo_arquetipo=cluster_a == cluster_b,
    )


comparacion = ComparacionJugadores(
    jugador_a=_estadisticas_de(fila_a["player_id"], fila_a),
    jugador_b=_estadisticas_de(fila_b["player_id"], fila_b),
    grupo_comparacion_a=_grupo_comparacion_de(fila_a),
    grupo_comparacion_b=_grupo_comparacion_de(fila_b),
    percentiles=_percentiles_comparados_de(fila_a, fila_b),
    valor=_valor_comparado_de(fila_a, fila_b),
    arquetipo=_arquetipo_comparado_de(fila_a, fila_b),
)

print(f"{comparacion.jugador_a.nombre}: {comparacion.valor.banda_a.valor_medio:,.0f} €")
print(f"{comparacion.jugador_b.nombre}: {comparacion.valor.banda_b.valor_medio:,.0f} €")


Erling Haaland: 332,425,329 €
Kylian Mbappé: 88,417,891 €


## 3. Qué JSON ve el LLM

`_datos_para_prompt` (en `generador.py`) es el único punto de contacto entre el LLM y los
datos: nunca le pasa el `ComparacionJugadores` completo (que trae estadísticas crudas,
`foto_url`, etc.), solo el resumen ya interpretado -- percentiles en escala 0-100, valores en
euros redondeados, y la diferencia de valor ya reducida a "qué factor favorece a quién". El
LLM no puede inventar una cifra que no esté acá porque no tiene acceso a nada más.

In [4]:
datos = _datos_para_prompt(comparacion)
print(json.dumps(datos, indent=2, ensure_ascii=False))


{
  "jugador_a": {
    "nombre": "Erling Haaland",
    "club": "Manchester City",
    "posicion": "Centre-Forward",
    "edad": 25.0,
    "temporada": "2025-2026"
  },
  "jugador_b": {
    "nombre": "Kylian Mbappé",
    "club": "Real Madrid",
    "posicion": "Centre-Forward",
    "edad": 26.0,
    "temporada": "2025-2026"
  },
  "misma_posicion": true,
  "percentiles_vs_su_posicion": [
    {
      "estadistica": "Goles por 90'",
      "percentil_a": 95.7,
      "percentil_b": 96.5
    },
    {
      "estadistica": "Asistencias por 90'",
      "percentil_a": 88.2,
      "percentil_b": 79.4
    },
    {
      "estadistica": "Tiros por 90'",
      "percentil_a": 87.5,
      "percentil_b": 95.1
    },
    {
      "estadistica": "% de tiros a puerta",
      "percentil_a": 76.9,
      "percentil_b": 68.9
    },
    {
      "estadistica": "Entradas ganadas",
      "percentil_a": 74.1,
      "percentil_b": 26.5
    },
    {
      "estadistica": "Intercepciones",
      "percentil_a": 76.2,
    

## 4. Diseño del prompt y del schema

- `SYSTEM_PROMPT` fija el rol ("analista de scouting") y, sobre todo, las restricciones: no
  inventar cifras, no calcular nada nuevo, tratar el valor de mercado como estimación (nunca
  "recomendación de compra/venta" -- la misma regla de bandas del spec), y aclarar cuando los
  jugadores juegan posiciones distintas antes de comparar percentiles.
- `RESPONSE_SCHEMA` usa `response_format: json_schema` con `strict: True` -- mismo patrón
  validado en el prototipo de noticias (`clasificacion_noticia` en la versión anterior de
  este notebook), acá para forzar cuatro campos fijos (`resumen`,
  `fortalezas_jugador_a/b`, `explicacion_diferencia_valor`) en vez de dejar al modelo
  redactar en formato libre.

In [5]:
print(SYSTEM_PROMPT)


Eres un analista de scouting de fútbol. Se te entrega una comparación ya calculada entre dos jugadores: percentiles frente a jugadores de su misma posición y temporada, la contribución de un modelo de valor de mercado a la diferencia de valor entre ambos, y su arquetipo de estilo de juego (de un modelo de clustering). Tu único trabajo es REDACTAR esa comparación en español de forma clara, objetiva y CONCISA para un scout: ve directo a lo importante (fortalezas de cada uno y por qué difiere el valor), sin relleno ni frases genéricas.

Reglas estrictas:
- No inventes cifras, estadísticas ni hechos que no estén en los datos entregados.
- No calcules nada nuevo: limita tus afirmaciones a lo que los números ya indican.
- El valor de mercado es una estimación de un modelo estadístico, no una tasación oficial ni una recomendación de compra o venta: trátalo siempre como estimación.
- Si los dos jugadores juegan posiciones distintas, acláralo antes de comparar percentiles.
- Cada factor de 'fac

In [6]:
print(json.dumps(RESPONSE_SCHEMA, indent=2, ensure_ascii=False))


{
  "name": "narrativa_comparacion",
  "schema": {
    "type": "object",
    "properties": {
      "resumen": {
        "type": "string",
        "description": "1-2 oraciones que sitúan la comparación (posición y arquetipo de cada uno, y si son directamente comparables). Nada más -- el detalle va en fortalezas y en la explicación de valor, no acá."
      },
      "fortalezas_jugador_a": {
        "type": "array",
        "items": {
          "type": "string"
        },
        "description": "2 a 3 puntos donde el jugador A destaca sobre el B. Cada punto: una sola oración, directa, que nombre la estadística y qué implica."
      },
      "fortalezas_jugador_b": {
        "type": "array",
        "items": {
          "type": "string"
        },
        "description": "2 a 3 puntos donde el jugador B destaca sobre el A. Cada punto: una sola oración, directa, que nombre la estadística y qué implica."
      },
      "explicacion_diferencia_valor": {
        "type": "string",
        "desc

## 5. Bug encontrado probando con la API real: atribución cruzada

Documentado en AGENTS.md (decisiones de diseño del Comparador + LLM, punto 4). La primera
versión de `_datos_para_prompt` mandaba un booleano (`favorece_a`) para indicar a qué jugador
beneficiaba cada factor de valor -- el LLM tenía que resolver la indirección
booleano → "A" o "B" → nombre del jugador, y en pruebas repetidas a veces le asignaba el
factor al jugador equivocado en la prosa.

El fix fue quitarle esa indirección al modelo: en vez de un booleano, `_datos_para_prompt` ya
manda el nombre del jugador resuelto (`jugador_favorecido`, ver la celda de la sección 3
arriba) -- literalmente el string que el LLM debe copiar, no algo que tenga que derivar.

In [7]:
# Antes (causaba atribución cruzada en pruebas repetidas):
factor_antes = {"factor": "Fuerza histórica del equipo", "favorece_a": True}
# El LLM tenía que resolver: True -> "a" -> comparacion.jugador_a.nombre

# Después (lo que corre en producción, ver _datos_para_prompt arriba):
factor_despues = {"factor": "Fuerza histórica del equipo", "jugador_favorecido": "Erling Haaland"}
# Sin indirección: el LLM solo copia el nombre que ya viene resuelto.


## 6. Llamar al modelo real

`NarradorComparacion.redactar()` es la clase que instancia `app/main.py` en
`app.state.narrador` (`None` si falta `OPENAI_API_KEY` -> el endpoint responde 503, nunca
inventa una narrativa sin el LLM configurado). Acá se llama igual que lo haría
`GET /jugadores/comparar/narrativa`.

In [8]:
narrador = NarradorComparacion(api_key=client_api_key)
resultado = narrador.redactar(comparacion)
print(json.dumps(resultado, indent=2, ensure_ascii=False))


{
  "resumen": "Erling Haaland y Kylian Mbappé, ambos delanteros centro con el arquetipo de 'estrella ofensiva de máximo volumen', son comparables directamente tanto por posición como por rol.",
  "fortalezas_jugador_a": [
    "Haaland supera a Mbappé en asistencias por 90', mostrando mayor aporte asociativo en la posición.",
    "Haaland tiene un porcentaje más alto de tiros a puerta, reflejando mejor precisión o selección de remates.",
    "Haaland destaca más en entradas ganadas e intercepciones, contribuyendo más en presión y recuperación defensiva."
  ],
  "fortalezas_jugador_b": [
    "Mbappé lidera en goles por 90', situándose en el percentil superior incluso frente a Haaland.",
    "Mbappé ejecuta más tiros por 90', indicando volumen ofensivo y protagonismo en el ataque.",
    "Mbappé tiene un altísimo percentil en centros, sumando capacidad de desborde o aporte desde banda."
  ],
  "explicacion_diferencia_valor": "La estimación de valor de mercado favorece ampliamente a Haalan

## 7. Validar contra el schema de respuesta de la app

`NarrativaComparacion` (en `app/src/app/schemas/comparacion.py`) es el `response_model` real
del endpoint -- si el dict que devuelve el LLM no calza, esto revienta acá mismo, antes de
llegar al frontend. También agrega `metodologia_nota` (la misma nota reutilizada de
`PrediccionValor`, ver AGENTS.md), que el LLM nunca redacta -- es texto fijo del backend.

In [9]:
narrativa = NarrativaComparacion(**resultado)
print(narrativa.metodologia_nota)


Estimación de un modelo estadístico entrenado con datos históricos de rendimiento y valor de mercado. No es una tasación oficial ni una recomendación de compra o venta: preséntese siempre como rango, nunca como cifra puntual.


## 8. Decisiones de producto ya tomadas (detalle en AGENTS.md)

- Endpoint separado (`/comparar/narrativa`), disparado por un botón explícito en el frontend,
  no automáticamente al elegir dos jugadores -- la llamada a OpenAI tiene costo y puede tardar
  ~5-40s de punta a punta contra la API real.
- `app.state.narrador` es `None` si no hay `OPENAI_API_KEY` configurado; el endpoint responde
  503 en vez de degradar a una narrativa sin LLM.
- El LLM nunca recibe estadísticas crudas ni hace ningún cálculo -- todo lo que redacta ya
  viene resuelto en `datos` (sección 3). Si algún día se le agrega un campo nuevo al prompt,
  debe ser un valor ya calculado por la app, nunca algo que el LLM tenga que derivar (la
  lección del bug de la sección 5).